# DCA vs Lump Sum, Backtested
You have 10,000 to put into Bitcoin. All of it today (lump sum), or in equal parts over the coming
months — DCA (dollar-cost averaging)? We test both on every possible start day since August 2017,
not on one chart picked to make a point, and compare with the study most of the internet quotes:
Vanguard, "Cost averaging: invest now or temporarily hold your cash?" (Finlay & Zorn, February 2023).
Their result for stocks: lump sum won 68% of the time. Education, not financial advice.

Our settings, as in Vanguard's study: the wealth after one year, a new start every day, cash waiting
to be invested earns nothing; spot, a 0.1% fee per purchase; USDT (Tether, a dollar stablecoin).

## 1. Helpers from episode q00
`binance_vision` — raw Binance data with SHA-256 hash checks (episode q00). No freqtrade this time: a buying schedule is not a trading rule, and a few lines of numpy say it exactly.

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py
from binance_vision import download_all, months
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Bitcoin and Ethereum, every day
Daily spot candles from August 2017, when Binance's archive starts, to October 2026, each monthly file checked against its SHA-256 hash (`download_all`, episode q00). From January 2025 spot timestamps are in microseconds. Uses `download_all`, `months` (section 1).

In [ ]:
def daily(coin):
    url = f"https://data.binance.vision/data/spot/monthly/klines/{coin}USDT/1d"
    parts = download_all([f"{url}/{coin}USDT-1d-{m}.zip"
                          for m in months("2017-08", "2026-10")])
    d = pd.concat([p.iloc[:, :5] for p in parts], ignore_index=True)
    d.columns = ["date", "open", "high", "low", "close"]
    ms = np.where(d["date"] > 1e14, d["date"] // 1000, d["date"])  # us -> ms
    d["date"] = pd.to_datetime(ms, unit="ms", utc=True)
    return d.drop_duplicates("date").set_index("date").sort_index()

btc, eth = daily("BTC"), daily("ETH")
print(f"{len(btc):,} days, {btc.index[0].date()} to {btc.index[-1].date()}")
print(f"Bitcoin {btc['open'].iloc[0]:,.0f} -> {btc['close'].iloc[-1]:,.0f}")

## 3. One start, by hand
17 December 2017, the top of Bitcoin's 2017 rally. 10,000 USDT (Tether, a dollar stablecoin) all at once, against DCA (dollar-cost averaging): twelve equal parts of 833.33, one every 30 days. Both are valued one year after the start. A 0.1% fee on every purchase. Uses `btc` (section 2).

In [ ]:
FEE = 0.001                                       # 0.1% of every purchase
start = btc.index.get_loc(pd.Timestamp("2017-12-17", tz="UTC"))
buys = btc["open"].iloc[start:start + 360:30]     # 12 buys, 30 days apart
coins = 10_000 / 12 * (1 - FEE) / buys            # coins bought each time
table = pd.DataFrame({"price": buys.round().astype(int),
                      "coins": coins.round(4)})
table.index = table.index.date                    # the day of each buy
print(table.to_string())
lump = 10_000 * (1 - FEE) / buys.iloc[0]          # everything on day 1
end = btc["close"].iloc[start + 365]              # one year later
print(f"lump sum: {lump:.4f} BTC, DCA: {coins.sum():.4f} BTC")
print(f"a year later at {end:,.0f}: lump sum {lump * end:,.0f},"
      f" DCA {coins.sum() * end:,.0f} USDT")

## 4. Why: the harmonic mean
DCA (dollar-cost averaging) spends the same money every time, so it buys more coins when the price is low: its cost per coin is the harmonic mean $H = \dfrac{n}{\sum 1/p_i}$, which is never above the plain average. Lump sum wins exactly when the first price $p_1$ is below $H$: coins of lump sum ÷ coins of DCA $= H / p_1$. Uses `buys`, `lump`, `coins` (section 3).

In [ ]:
H = len(buys) / (1 / buys).sum()                  # harmonic mean price
print(f"plain average {buys.mean():,.0f}, harmonic mean {H:,.0f},"
      f" first price {buys.iloc[0]:,.0f}")
print(f"H / first price = {H / buys.iloc[0]:.4f};"
      f" lump / DCA coins = {lump / coins.sum():.4f}")

## 5. Every start day at once
A plan = the days, counted from the start, on which an equal part is bought; lump sum is the plan with one day, 0. Each row is one start day: what 1 USDT (Tether, a dollar stablecoin) became on each of the 366 days of its year (cash still waiting + coins at that day's close). Uses `FEE` (3), `btc` (2).

In [ ]:
def year_paths(d, plan, days=365):
    o, c = d["open"].to_numpy(), d["close"].to_numpy()
    n = len(d) - days                         # starts with a full year after
    buy = np.zeros(days + 1)
    buy[list(plan)] = 1 / len(plan)           # share of the money per buy
    paths = np.empty((n, days + 1))
    for i in range(n):
        coins = np.cumsum(buy * (1 - FEE) / o[i:i + days + 1])
        cash = 1 - np.cumsum(buy)             # money not yet invested
        paths[i] = cash + coins * c[i:i + days + 1]
    return pd.DataFrame(paths, index=d.index[:n])

lump_btc = year_paths(btc, [0])
dca_btc = year_paths(btc, range(0, 360, 30))      # 12 parts, 30 days apart
print(f"{len(lump_btc):,} start days,"
      f" {lump_btc.index[0].date()} to {lump_btc.index[-1].date()}")
print("17 Dec 2017, after a year:",
      f"{10_000 * lump_btc.loc['2017-12-17', 365]:,.0f} and",
      f"{10_000 * dca_btc.loc['2017-12-17', 365]:,.0f} USDT")

## 6. Who wins, over 2,998 start days
After one year, as in Vanguard's study. Uses `lump_btc`, `dca_btc` (section 5).

In [ ]:
a, b = lump_btc[365], dca_btc[365]          # 1 USDT after a year
print(f"lump sum wins on {(a > b).mean():.1%} of the start days")
for name, v in (("lump sum", a), ("DCA 12 months", b)):
    print(f"{name:>13}: median {v.median():.2f}"
          f" | worst {v.min():.2f} {v.idxmin().date()}"
          f" | best {v.max():.2f} {v.idxmax().date()}")
print(f"average gap: lump sum ahead by {(a - b).mean():+.2f} per 1 USDT")

## 7. Shorter and longer plans
Vanguard's base case: DCA (dollar-cost averaging) over 3 months, three parts a month apart. Then DCA over 6 and 12 months, and over 52 weeks — a part every Monday. Uses `year_paths` (5), `btc` (2), `a` (6).

In [ ]:
plans = {"DCA 3 months": range(0, 90, 30),
         "DCA 6 months": range(0, 180, 30),
         "DCA 12 months": range(0, 360, 30),
         "DCA 52 weeks": range(0, 364, 7)}
ends = {"lump sum": a}
for name, plan in plans.items():
    ends[name] = year_paths(btc, plan)[365]
    print(f"{name:>13}: lump sum wins on {(a > ends[name]).mean():.1%},"
          f" median {ends[name].median():.2f} vs {a.median():.2f}")

## 8. How many years is that, really?
2,998 start days overlap: two starts a day apart share 364 of their 365 days. Taking every 365th start day leaves years that share no day at all: only nine of them. Uses `lump_btc`, `dca_btc` (section 5).

In [ ]:
starts = lump_btc.index[::365]          # every 365th day: no shared days
apart = pd.DataFrame({"lump sum": lump_btc.loc[starts, 365],
                      "DCA 12 months": dca_btc.loc[starts, 365]})
apart["winner"] = np.where(apart["lump sum"] > apart["DCA 12 months"],
                           "lump sum", "DCA")
apart.index = apart.index.date
print(apart.round(2).to_string())

## 9. Ethereum, a second coin
The same plans on Ethereum, the same 2,998 start days. Uses `year_paths` (5), `eth` (2), `plans` (7).

In [ ]:
e = year_paths(eth, [0])[365]
for name, plan in plans.items():
    v = year_paths(eth, plan)[365]
    print(f"{name:>13}: lump sum wins {(e > v).mean():.1%},"
          f" median {v.median():.2f} vs {e.median():.2f},"
          f" mean {v.mean():.2f} vs {e.mean():.2f}")

## 10. The pain on the way
Inside the year: the deepest fall from a peak, and the worst moment below the money put in. Median over the 2,998 start days, and the worst one. Uses `lump_btc`, `dca_btc` (section 5).

In [ ]:
def falls(paths):
    peak = paths.cummax(axis=1)
    deepest = (1 - paths / peak).max(axis=1)      # from the best day so far
    under = (1 - paths.min(axis=1)).clip(lower=0)  # below the money put in
    return deepest, under

for name, p in (("lump sum", lump_btc), ("DCA 12 months", dca_btc)):
    deepest, under = falls(p)
    print(f"{name:>13}: deepest fall median {deepest.median():.0%},"
          f" worst {deepest.max():.0%}; below cost worst {under.max():.0%}")

## 11. The best and the worst start
17 December 2017, the worst day to start a lump sum, and 13 March 2020, the best: the covid crash. 10,000 USDT (Tether, a dollar stablecoin), day by day for a year. Uses `lump_btc`, `dca_btc` (section 5).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, day in zip(ax, ("2017-12-17", "2020-03-13")):
    for name, p in (("lump sum", lump_btc), ("DCA 12 months", dca_btc)):
        path = 10_000 * p.loc[day]
        axis.plot(path.index, path, label=name)
        print(f"{day} {name:>13}: lowest {path.min():>6,.0f},"
              f" after a year {path.iloc[-1]:>7,.0f}")
    axis.set_title(f"start {day}"); axis.set_xlabel("days")
ax[0].legend(); plt.show()

## 12. The Sharpe ratio: return per unit of wobble
The same four lines as episode q03: Sharpe ratio $S = \dfrac{\text{mean of the daily returns}}{\text{their standard deviation}} \times \sqrt{365}$; risk-free rate 0, as waiting cash earns nothing here. One Sharpe per start day and plan, over its year. Uses `lump_btc`, `dca_btc` (5).

In [ ]:
def sharpe(curve):
    day = curve.resample("1D").last()           # the account once a day
    r = day.pct_change().dropna()               # daily returns
    return r.mean() / r.std() * 365 ** 0.5      # a year = 365 crypto days

def sharpes(paths):
    days = pd.date_range("2000-01-01", periods=paths.shape[1], freq="D")
    return pd.Series([sharpe(pd.Series(row, index=days))
                      for row in paths.to_numpy()], index=paths.index)

s_lump, s_dca = sharpes(lump_btc), sharpes(dca_btc)
for name, s in (("lump sum", s_lump), ("DCA 12 months", s_dca)):
    print(f"{name:>13}: median Sharpe {s.median():.2f},"
          f" above 1 on {(s > 1).mean():.0%} of the start days")
print(f"lump sum's Sharpe higher on {(s_lump > s_dca).mean():.0%}")

## 13. Everything side by side
Bitcoin, 2,998 start days, one row per plan; a legend explains every column. Uses `year_paths` (5), `btc` (2), `plans` (7), `ends`, `a` (7), `falls` (10), `sharpes` (12).

In [ ]:
rows = {}
for name, plan in {"lump sum": [0], **plans}.items():
    p = year_paths(btc, plan)
    deepest, under = falls(p)
    beats = "-" if name == "lump sum" else f"{(p[365] > a).mean():.0%}"
    rows[name] = {"beats lump": beats,
                  "median": f"{p[365].median():.2f}",
                  "worst": f"{p[365].min():.2f}",
                  "fall": f"{deepest.median():.0%}",
                  "below": f"{under.max():.0%}",
                  "Sharpe": f"{sharpes(p).median():.2f}"}
print(pd.DataFrame(rows).T.to_string())
print("money: what 1 USDT (Tether, a dollar stablecoin) became in a year")
print("beats lump: share of start days the plan ended above lump sum")
print("median = the middle start day, ranked from worst to best")
print("worst = the lowest of the 2,998 start days, Aug 2017 - Oct 2025")
print("fall: median deepest fall from a peak inside the year")
print("below: the worst moment below the money put in")
print("Sharpe: median; mean daily return / its std x sqrt(365);")
print("  above 1 is the usual target")